## EDA on label from User

QUESTION 1: Find all possible unique values
- Category
- Subcategory
- Root_code
- Sub_code

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel('label.xlsx')

print("--- 1. UNIQUE VALUES IN DATASET ---")
print(f"Categories ({df['Category'].nunique()}): {df['Category'].dropna().unique()}")
print(f"Subcategories ({df['Subcategory'].nunique()}): {df['Subcategory'].dropna().unique()}")
print(f"Root Codes ({df['root_code'].nunique()}): {df['root_code'].dropna().unique()}")
print(f"Sub Codes ({df['sub_code'].nunique()}): {df['sub_code'].dropna().unique()}\n")

QUESTION 2: Find mappings and human errors

In [ ]:
print("--- 2. MAPPING RULES & ANOMALY DETECTION ---")

# Group by the human text labels and see what codes they mapped to, counting occurrences
mapping_df = df.groupby(
    ['Category', 'Subcategory', 'root_code', 'sub_code'], 
    dropna=False
).size().reset_index(name='Count')

# Sort so that for each Category/Subcategory, the most frequent mapping is at the top
mapping_df = mapping_df.sort_values(
    by=['Category', 'Subcategory', 'Count'], 
    ascending=[True, True, False]
)

# Calculate the total times a Category/Subcategory pair appears
mapping_df['Total_Category_Occurrences'] = mapping_df.groupby(['Category', 'Subcategory'])['Count'].transform('sum')

# Calculate the percentage of times THIS specific mapping was used
mapping_df['Mapping_Confidence_%'] = (mapping_df['Count'] / mapping_df['Total_Category_Occurrences']) * 100

print("\nAll Mappings (Sorted by frequency):")
display(mapping_df) # Use print(mapping_df) if not in Jupyter

In [ ]:
# --- ISOLATING THE ERRORS ---
# Assuming that whatever mapping happens the majority of the time (>50%) is the "Correct" rule...
# Anything else is likely a human annotation error.

print("\n--- LIKELY HUMAN ANNOTATION ERRORS ---")
# Filter for mappings that happen less than 20% of the time for that specific category
anomalies = mapping_df[mapping_df['Mapping_Confidence_%'] < 20.0]

if anomalies.empty:
    print("Good news! No major anomalies found. The mapping is consistent.")
else:
    print("Found potential mislabels! Annotators likely messed these up:")
    display(anomalies)

In [ ]:
# ==========================================
# CELL 1: EYEBALL SPECIFIC CATEGORIES
# ==========================================
from IPython.display import Image, display
import os

# 1. Define the specific pair you want to investigate
TARGET_CATEGORY = "Insert Category Here"       # e.g., "Medical Document"
TARGET_SUBCAT = "Insert Subcategory Here"      # e.g., "Lab Results"

# 2. Filter the existing 'df' for this specific pair
subset_df = df[(df['Category'] == TARGET_CATEGORY) & (df['Subcategory'] == TARGET_SUBCAT)]

print(f"Found {len(subset_df)} documents for {TARGET_CATEGORY} -> {TARGET_SUBCAT}\n")

# 3. Display the images (Showing the first 10 to avoid crashing the notebook)
for idx, row in subset_df.head(10).iterrows():
    print("-" * 50)
    print(f"Row Index: {idx}")
    print(f"Current Labels -> root_code: [{row['root_code']}] | sub_code: [{row['sub_code']}]")
    print(f"Filepath: {row['Filepath']}")
    
    filepath = str(row['Filepath'])
    if os.path.exists(filepath):
        # Display the image directly in the notebook output
        display(Image(filename=filepath, width=600))
    else:
        print(f"Image not found at path: {filepath}")

In [ ]:
# ==========================================
# CELL 2: BATCH REPLACE & SAVE
# ==========================================

# 1. Define the pair you want to fix
TARGET_CATEGORY = "Insert Category Here"
TARGET_SUBCAT = "Insert Subcategory Here"

# 2. Define the CORRECT codes they should be mapped to
CORRECT_ROOT_CODE = "NON"
CORRECT_SUB_CODE = "OTH"

# 3. Create the filter condition
condition = (df['Category'] == TARGET_CATEGORY) & (df['Subcategory'] == TARGET_SUBCAT)

# 4. Use .loc to safely update the values in the original dataframe
df.loc[condition, 'root_code'] = CORRECT_ROOT_CODE
df.loc[condition, 'sub_code'] = CORRECT_SUB_CODE

print(f"Successfully updated {condition.sum()} rows to {CORRECT_ROOT_CODE} -> {CORRECT_SUB_CODE}.")

# 5. Export the cleaned data to CSV
output_filename = 'label_cleaned.csv'
df.to_csv(output_filename, index=False)
print(f"Cleaned dataset exported as: {output_filename}")

## Evaluation

In [ ]:
# ==========================================
# v13 CHANGELOG (vs v12) -- REVERT TWO DECISIONS BEFORE THEY SHIP
# ==========================================
# 1. All receipts (medical AND non-medical) now route to financial_receipt. Removed
#    medical_receipt entirely -- reverts the edit to AGENT3_SYS_PROMPT, so that prompt
#    is back to exactly what your senior gave you, no sync needed.
# 2. Removed not_for_underwriting_subtag. not_for_underwriting is now a single flat
#    terminal class -- no eform/correspondence/envelope/portrait/illegible breakdown.
#    The descriptive examples stay in the prompt (still useful for the model to
#    recognize the pattern), just not captured as a structured output field anymore.
# ==========================================


import math
import time
import random
import os
import csv
import threading
import pandas as pd
from typing import Literal, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
from pydantic import BaseModel
from openai import AzureOpenAI, BadRequestError, RateLimitError, APIConnectionError, APITimeoutError

try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

# ==========================================
# 1. DEFINE STRUCTURED OUTPUT SCHEMAS
# ==========================================

class MacroClassifierOutput(BaseModel):
    chain_of_thought: str
    macro_class: Literal["medical", "financial", "identification", "not_for_underwriting"]


class FinancialSpecialistOutput(BaseModel):
    chain_of_thought: str
    subcategory: Literal[
        "financial_bankstatement", "financial_bookbank", "financial_companyregistration",
        "financial_selfincomedeclaration", "financial_receipt", "financial_others"
    ]


class IdentificationSpecialistOutput(BaseModel):
    chain_of_thought: str
    subcategory: Literal[
        "id_driverlicense", "id_fatca_w9", "id_foreignerconfirmationform",
        "id_foreigner_nationalid", "id_visastamp", "id_passport",
        "id_statelessid", "id_thainationalid", "id_workpermit",
        "id_houseregistration", "id_marriagecertificate",  # still tentative, see prior note
        "id_others"
    ]


class MedSpecialistOutput(BaseModel):
    chain_of_thought: str
    subcategory: Literal[
        "medical_clinical", "medical_healthcheck",
        "medical_lab", "medical_others"
    ]

# ==========================================
# 2. DEFINE PROMPT VARIABLES
# ==========================================

# --- STEP 1: MACRO CLASSIFIER (replaces the old binary triage + 4-way router) ---
AGENT1_SYS_PROMPT = """You are a Document Macro Classifier for an insurance underwriting pipeline. Classify raw OCR/markdown text into exactly one of: `medical`, `financial`, `identification`, `not_for_underwriting`. Treat the patterns below as supporting evidence, not an exact-string checklist -- read holistically. Ignore PII.
 
### Ignore Watermarks
Insurance disclaimers (+ date/time stamps) are never evidence of document type -- ignore them completely.
 
### medical
Route here for genuine clinical content: clinical notes, lab metrics, health-check parameters, prescriptions, test ranges, normal/abnormal flags. Garbled OCR with technical-looking terms next to numbers/units/ranges still counts as low-confidence medical (flag the uncertainty).
Do NOT route here for a hospital/pharmacy receipt or billing statement -- those are financial_receipt regardless of medical context (see financial, below). Do NOT route here for correspondence that merely discusses or requests medical information without containing the clinical data itself -- that's not_for_underwriting.
 
### financial
- Bank/account identity (account number + bank name/branch), or a native transaction ledger (dated money movements, ideally a markdown table).
- Corporate registration number + certification/signature block.
- Self income declaration: the person reporting, clarifying, or declaring their own income or source of income -- even if laid out as a form or questionnaire.
- Any receipt or proof-of-payment document -- including hospital/pharmacy/medical treatment receipts (ค่ายา, ค่ารักษา, ค่าห้อง), government license/permit fee receipts, retail receipts, or any other billing statement. All receipts are financial, regardless of what the payment was for.
- General agreements/contracts: lease, sale, or other business contracts.
 
### identification
A genuine ID/travel/civil-registry document with its own native structure (not a field filled into someone else's form):
- Issuing country name/code (e.g. "LAO PDR", "RDP LAO"), a genuine name+DOB or name+issue/expiry block, "รับรองสำเนาถูกต้อง" (certified true copy), or a `<figure>` region actually wrapping such a personal-detail block -- not just a name/reference number sitting near a `<figure>` tag with no other identity structure.
- Stateless-ID text, or garbled non-Thai script (e.g. Lao misread as Thai) combined with a country-code header.
- MRZ lines (letters/digits + long "<" runs, e.g. "PA123SD464987<<<<<<<"), or immigration/visa keywords -- "VISA", "VISA CLASS"/"VISACLASS", "IMMIGRATION", "DEPARTED", "ADMITTED", "ENTRY" -- near messy digits/dates. Standardized terms, safe to trust literally even amid heavy OCR noise; don't let a "looks like noise" impression override them.
- FATCA/W-9 tax-withholding declaration forms.
- Work permit documents: labor authorization booklets, employer/employee details, type-of-work (ประเภทงาน) fields, Department of Employment issuing text. This ALSO includes work permit renewals and amendment/endorsement documents such as "Change or addition of category of work" (รายการเปลี่ยนเพิ่มประเภทงาน) -- these are still the same underlying document type, just an update to it, not a new document category.
- Household registration (ทะเบียนบ้าน): household registration book header, house/address registry formatting, list of household members.
- Marriage certificate (ใบสำคัญการสมรส / ทะเบียนสมรส): marriage registration terminology, groom/bride names, registrar signature, marriage date.
- If the document is clearly a personal ID-type card/document (has a photo placeholder, ID number, name, and an issuing context) but doesn't match any pattern above -- e.g. a student ID card, employee badge, or other institutional ID -- it STILL counts as `identification` here. Don't route it to not_for_underwriting just because it isn't one of the specific types listed; the specialist downstream has its own catch-all for exactly this case.
- Reminder: watermark text (see Global Rules above) applies here too. An ID document with an insurance watermark/date stamped near it is still identification -- don't let the watermark, or a date sitting close to it, push you toward not_for_underwriting.
 
### not_for_underwriting
Default when nothing above applies -- the underwriter will not use this document for processing. This includes several distinct patterns, all landing in this one class:
- Insurance application/policy paperwork (eForm) with no independent financial or identity substance -- fields filled in about an applicant (เบี้ยประกัน, policy details), documents opening with "บันทึกคำชี้แจงเกี่ยวกับใบคำขอประกันภัย", or consent letters for data collection. Litmus test: strip out the applicant's name/ID/policy number -- is any standalone financial, clinical, or identity statement left? If nothing remains but "applying for / updating / requesting details about a policy," it belongs here, regardless of medical or financial-sounding language elsewhere.
- Correspondence: letters or memos between the insured and the underwriter requesting or forwarding additional documents/information -- administrative communication, not a document in its own right.
- Envelope/mailing metadata -- sender name, a policy-style reference code (e.g. "T1348646"), addressing text. Describes or routes a document without containing it.
- A portrait photo or ID placeholder with no accompanying issuing text.
- Illegible noise, or genuinely unclear content that doesn't fit any of the above.
 
In chain_of_thought: name the pattern found, and if not_for_underwriting, briefly say which of the patterns above it resembles."""
 
AGENT1_USER_PROMPT = "Classify the following raw OCR text into medical, financial, identification, or not_for_underwriting:\n{ocr_text}"


# --- STEP 2a: FINANCIAL SPECIALIST ---
AGENT_FINANCIAL_SYS_PROMPT = """You are an expert Financial Document Specialist. You receive text already confirmed as financial. Perform granular classification into one of six leaf classes. Use markdown structure (tables, headers) where present. Do not process PII.

- financial_bankstatement: running transactional ledgers, money transfers, account balances, mobile banking markers -- ideally a markdown table of dated transaction rows.
- financial_bookbank: static account identity headers (bilingual accounts, branches, bank names) without a long transaction ledger.
- financial_companyregistration: formal commercial corporate registry verbiage and certification signatures.
- financial_selfincomedeclaration: the person reporting, clarifying, or declaring their own income, salary, or source of income -- regardless of whether it's laid out as a form or questionnaire.
- financial_receipt: any proof-of-payment document -- hospital/pharmacy/medical treatment receipts, government license/permit fee receipts, retail receipts, or any other billing statement, regardless of what the payment was for.
- financial_others: financial in nature but doesn't cleanly match the above -- also covers general agreements/contracts (lease, sale, business contracts).

Provide step-by-step structural rationale in chain_of_thought before selecting the final leaf subcategory."""

AGENT_FINANCIAL_USER_PROMPT = "Perform deep financial classification on this verified text:\n{ocr_text}"


# --- STEP 2b: IDENTIFICATION SPECIALIST ---
AGENT_ID_SYS_PROMPT = """You are an expert Identification Document Specialist. You receive text already confirmed as an identification/travel document. Perform granular classification. Use markdown structure where present. Do not process PII.

- id_thainationalid / id_foreigner_nationalid: national demographic card headers and identity issuing text blocks. Foreign-script documents (e.g. a Lao national ID) still count even if largely unreadable by OCR -- look for legible fragments like issuing country name/code, name, or a DOB pattern.
- id_passport: the passport bio data page -- photo/data block, passport number, nationality, an MRZ line (letters/digits + long "<" runs).
- id_visastamp: a Thai visa page and/or immigration control stamp -- visa category codes, "DEPARTED"/"ADMITTED"/"ENTRY" keywords, and (often garbled, since stamp text is curved) date-like digit strings near them.
- id_workpermit: labor authorization rules and official employment permission context.
- id_fatca_w9: US tax withholding terminology, backup withholding rules, entity declaration sections.
- id_foreignerconfirmationform: an official government-issued confirmation/declaration form attesting to a foreign national's identity or status -- issued BY a government/authority ABOUT the person, not filled in by an agent for a policy application.
- id_statelessid: a stateless-person identity card -- card layout similar to a national ID (photo placeholder, ID number, name, DOB) but with issuing text indicating stateless/no-registered-nationality status.
- id_driverlicense: driver's license.
- id_houseregistration: household registration document (ทะเบียนบ้าน) -- household registration book header, house/address registry formatting, list of household members.
- id_marriagecertificate: marriage certificate (ใบทะเบียนสมรส) -- marriage registration terminology, groom/bride names, registrar signature, marriage date.
- id_others: clearly an ID-type artifact that doesn't cleanly match any subtype above. Expected, valid outcome for borderline documents -- don't force a wrong specific subtype. NOTE: audit this bucket periodically -- some things that land here (e.g. a license/permit renewal fee) may actually be a mislabeled receipt (financial_receipt) rather than a true identification document; if the content is fundamentally about a payment/fee rather than proving identity, flag this in chain_of_thought instead of forcing it here.

Provide step-by-step structural rationale in chain_of_thought before selecting the final leaf subcategory."""

AGENT_ID_USER_PROMPT = "Perform deep identification classification on this verified text:\n{ocr_text}"


# --- MEDICAL SPECIALIST PROMPTS (UNCHANGED -- owned by a different prompt owner) ---
AGENT3_SYS_PROMPT = """You are a strict Medical Document Specialist. Categorize pre-verified medical OCR text into the exact clinical subclass. Focus on clinical structures and specific medical anchors.
 
### Subclass Priority & Evaluation Guides
Evaluate in this priority order -- if multiple patterns are present on the same page, the higher-priority match wins.
 
- **medical_lab (highest priority):** Laboratory test results. Contains tabular test data with columns for Test Name, Result, Units, and Reference Range.
  Keywords: CBC, BUN, Creatinine, Lipid Profile, Glucose, mg/dL, mmol/L, x10^3/uL.
  NOT included: waveform studies, signal recordings, or functional diagnostics (e.g., ECG, EKG, EMG, EEG, spirometry graphs, imaging reports) -- these go to medical_clinical instead (see below), unless the same page also carries extractable lab values, in which case medical_lab still wins.
 
- **medical_healthcheck (second priority):** Health check / physical examination data. Any page containing extractable vital signs or physical examination findings -- Blood Pressure, BMI, Heart Rate, Weight, Height, or a structured wellness summary. Applies regardless of whether the source document is a dedicated checkup report (e.g., "Annual Health Checkup", "Executive Health Screening") or a clinical encounter record (OPD/IPD) that includes a Physical Examination (PE) section with structured vitals.
 
- **medical_clinical (third priority):** Everything genuinely medical that isn't extractable lab data or extractable vitals. This is deliberately a broad category: medical_lab and medical_healthcheck feed a downstream extraction node that pulls structured values (test results, BMI, BP, weight) -- medical_clinical is the landing spot for real clinical content that node doesn't need structured values from. Includes:
  - Clinical narrative/notes: doctor consultation notes, progress notes, admission or discharge summaries, diagnosis lists, prescriptions, treatment plans, operative notes, hospital course documentation. Typically has visit/admission dates, narrative assessments, plans, and medication orders.
  - Pathology reports.
  - Imaging or functional diagnostic reports with no extractable lab or vitals data on the page -- X-ray, ultrasound, ECG/EKG, EMG, EEG, spirometry, or similar studies.
  NOT included: pages whose primary content is a table of vital signs or PE measurements (-> medical_healthcheck) or a lab results table (-> medical_lab).
 
- **medical_others:** Medical documents that don't structurally fit any category above.
  SPECIFIC INCLUSION: always route Sleep Test reports here.
 
In chain_of_thought: state the specific clinical anchors found and explicitly name the priority logic applied -- e.g. "Found lab table with mg/dL, overriding vitals", "Found BP/BMI but no labs", "X-ray report, no lab/vitals values present -> clinical", "Sleep test mention -> others" -- before making your final selection."""
 
AGENT3_USER_PROMPT = "Perform deep clinical classification on this verified medical text:\n{ocr_text}"

# ==========================================
# 3. CORE COGNITIVE PIPELINE
# ==========================================

def calculate_confidence(response_logprobs) -> float:
    """Whole-response confidence (legacy). Prefer calculate_field_confidence for
    anything you act on."""
    if not response_logprobs or not response_logprobs.content:
        return 0.0
    token_logprobs = [t.logprob for t in response_logprobs.content if t.logprob is not None]
    if not token_logprobs:
        return 0.0
    avg_logprob = sum(token_logprobs) / len(token_logprobs)
    return round(math.exp(avg_logprob) * 100, 2)


def calculate_field_confidence(response_logprobs, field_value: str) -> float:
    """Isolates and averages logprobs only for the tokens making up the classification
    enum value itself, rather than the whole JSON payload including chain_of_thought."""
    if not response_logprobs or not response_logprobs.content:
        return 0.0

    tokens = response_logprobs.content
    token_strs = [t.token for t in tokens]

    running = ""
    span_start, span_end = None, None
    for i, tok in enumerate(token_strs):
        prev_len = len(running)
        running += tok
        if field_value in running[max(0, prev_len - len(field_value)):]:
            end_idx = i
            partial = ""
            start_idx = i
            for j in range(i, -1, -1):
                partial = token_strs[j] + partial
                start_idx = j
                if field_value in partial:
                    break
            span_start, span_end = start_idx, end_idx

    if span_start is None:
        return calculate_confidence(response_logprobs)

    span_logprobs = [tokens[i].logprob for i in range(span_start, span_end + 1) if tokens[i].logprob is not None]
    if not span_logprobs:
        return calculate_confidence(response_logprobs)

    avg_logprob = sum(span_logprobs) / len(span_logprobs)
    return round(math.exp(avg_logprob) * 100, 2)


def call_with_retry(fn, *args, max_retries: int = 6, base_delay: float = 1.0,
                     max_delay: float = 60.0, **kwargs):
    """Wraps a single API call with retry + exponential backoff for 429s and transient
    connection errors. Respects Azure's Retry-After header when present."""
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except RateLimitError as e:
            retry_after = None
            try:
                retry_after = float(e.response.headers.get("retry-after"))
            except Exception:
                pass
            delay = retry_after if retry_after is not None else min(max_delay, base_delay * (2 ** attempt))
            delay += random.uniform(0, delay * 0.25)
            print(f"[RATE LIMIT] attempt {attempt + 1}/{max_retries}, sleeping {delay:.1f}s")
            time.sleep(delay)
        except (APIConnectionError, APITimeoutError) as e:
            delay = min(max_delay, base_delay * (2 ** attempt)) + random.uniform(0, 1)
            print(f"[TRANSIENT ERROR] {type(e).__name__}: {e} -- sleeping {delay:.1f}s")
            time.sleep(delay)
    raise RuntimeError(f"Exceeded max_retries ({max_retries}) calling {getattr(fn, '__name__', fn)}")


def run_cascade_pipeline(ocr_text: str, client: AzureOpenAI,
                          macro_model: str = "gpt-5-mini",
                          financial_model: str = "gpt-5",
                          id_model: str = "gpt-5",
                          med_model: str = "gpt-5") -> dict:
    """Executes the 2-stage architecture on a single piece of text.

    Step 1 (macro_model): single call, 4-way (medical / financial / identification /
        not_for_underwriting).
    Step 2 (financial_model / id_model / med_model): only called for medical / financial
        / identification. not_for_underwriting is terminal, flat, no subtag.
    """

    macro_response = call_with_retry(
        client.beta.chat.completions.parse,
        model=macro_model,
        messages=[
            {"role": "system", "content": AGENT1_SYS_PROMPT},
            {"role": "user", "content": AGENT1_USER_PROMPT.format(ocr_text=ocr_text)}
        ],
        response_format=MacroClassifierOutput,
        logprobs=True,
        top_logprobs=1
    )
    macro_data = macro_response.choices[0].message.parsed
    macro_class = macro_data.macro_class
    macro_conf = calculate_field_confidence(macro_response.choices[0].logprobs, macro_class)

    base_result = {
        "macro_decision": macro_class,
        "macro_reason": macro_data.chain_of_thought,
        "macro_confidence": macro_conf,
    }

    if macro_class == "medical":
        spec_response = call_with_retry(
            client.beta.chat.completions.parse,
            model=med_model,
            messages=[
                {"role": "system", "content": AGENT3_SYS_PROMPT},
                {"role": "user", "content": AGENT3_USER_PROMPT.format(ocr_text=ocr_text)}
            ],
            response_format=MedSpecialistOutput,
            logprobs=True,
            top_logprobs=1
        )
        spec_data = spec_response.choices[0].message.parsed
        spec_conf = calculate_field_confidence(spec_response.choices[0].logprobs, spec_data.subcategory)
        return {**base_result, "final_subcategory": spec_data.subcategory,
                "specialist_reason": spec_data.chain_of_thought, "specialist_confidence": spec_conf}

    if macro_class == "financial":
        spec_response = call_with_retry(
            client.beta.chat.completions.parse,
            model=financial_model,
            messages=[
                {"role": "system", "content": AGENT_FINANCIAL_SYS_PROMPT},
                {"role": "user", "content": AGENT_FINANCIAL_USER_PROMPT.format(ocr_text=ocr_text)}
            ],
            response_format=FinancialSpecialistOutput,
            logprobs=True,
            top_logprobs=1
        )
        spec_data = spec_response.choices[0].message.parsed
        spec_conf = calculate_field_confidence(spec_response.choices[0].logprobs, spec_data.subcategory)
        return {**base_result, "final_subcategory": spec_data.subcategory,
                "specialist_reason": spec_data.chain_of_thought, "specialist_confidence": spec_conf}

    if macro_class == "identification":
        spec_response = call_with_retry(
            client.beta.chat.completions.parse,
            model=id_model,
            messages=[
                {"role": "system", "content": AGENT_ID_SYS_PROMPT},
                {"role": "user", "content": AGENT_ID_USER_PROMPT.format(ocr_text=ocr_text)}
            ],
            response_format=IdentificationSpecialistOutput,
            logprobs=True,
            top_logprobs=1
        )
        spec_data = spec_response.choices[0].message.parsed
        spec_conf = calculate_field_confidence(spec_response.choices[0].logprobs, spec_data.subcategory)
        return {**base_result, "final_subcategory": spec_data.subcategory,
                "specialist_reason": spec_data.chain_of_thought, "specialist_confidence": spec_conf}

    # ---- Terminal: not_for_underwriting -- no Step 2 call ----
    return {
        **base_result,
        "final_subcategory": "not_for_underwriting",
        "specialist_reason": "Terminal at the macro classifier.",
        "specialist_confidence": macro_conf,
    }


# ==========================================
# 4. DATAFRAME EVALUATION EXECUTION
# ==========================================

_checkpoint_lock = threading.Lock()


def _process_one_row(idx, row, client, macro_model, financial_model, id_model, med_model):
    ocr_text = str(row['ocr_text'])

    try:
        res = run_cascade_pipeline(ocr_text, client, macro_model, financial_model, id_model, med_model)
    except BadRequestError as e:
        error_message = str(e).lower()
        if "content management policy" in error_message or "jailbreak" in error_message:
            print(f"[WARNING] Doc {idx + 1} blocked by Azure Jailbreak Filter. Skipping...")
            res = {
                "macro_decision": "blocked_by_firewall",
                "macro_reason": "Azure OpenAI Content Filter flagged this OCR text as a jailbreak attempt.",
                "macro_confidence": 0.0,
                "final_subcategory": "error_content_filter",
                "specialist_reason": "Skipped due to API security policy.",
                "specialist_confidence": 0.0
            }
        else:
            raise e
    except Exception as e:
        print(f"[ERROR] Doc {idx + 1} failed due to unexpected error: {e}")
        res = {
            "macro_decision": "api_error",
            "macro_reason": str(e),
            "macro_confidence": 0.0,
            "final_subcategory": "api_error",
            "specialist_reason": "API disconnected or failed.",
            "specialist_confidence": 0.0
        }

    combined_row = {
        "filename": row.get('filename'),
        "input_category_label": row.get('category'),
        "input_subcategory_label": row.get('subcategory'),
        **res
    }
    return idx, combined_row


def evaluate_experiment(csv_path: str, endpoint: str, api_key: str,
                         macro_model: str = "gpt-5-mini",
                         financial_model: str = "gpt-5",
                         id_model: str = "gpt-5",
                         med_model: str = "gpt-5",
                         max_workers: int = 5,
                         checkpoint_path: str = "experiment_full_cascade_results.csv",
                         resume: bool = True):
    """Reads the evaluation data, executes the cascade pipeline concurrently, and
    returns results.

    macro_model / financial_model / id_model / med_model: deployment names for each
        stage -- must match your Azure resource's deployment names.
    max_workers: number of documents processed in parallel. Start conservative (3-5) and
        raise it while watching for [RATE LIMIT] log lines.
    checkpoint_path: results are appended here as each document finishes.
    resume: if True and checkpoint_path already exists, filenames already present there
        are skipped instead of being re-processed.
    """

    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version="2024-08-01-preview"
    )

    df = pd.read_csv(csv_path)

    already_done = set()
    if resume and os.path.isfile(checkpoint_path):
        prior = pd.read_csv(checkpoint_path)
        if 'filename' in prior.columns:
            already_done = set(prior['filename'].dropna().astype(str))
        print(f"Resuming: {len(already_done)} document(s) already in checkpoint, will be skipped.")

    pending = [
        (idx, row) for idx, row in df.iterrows()
        if str(row.get('filename')) not in already_done
    ]

    print(f"Starting cascade experiment on {len(pending)} document(s) "
          f"(of {len(df)} total, {len(already_done)} already done) with max_workers={max_workers}...")

    fieldnames = None

    def write_row(combined_row):
        nonlocal fieldnames
        with _checkpoint_lock:
            file_exists = os.path.isfile(checkpoint_path)
            if fieldnames is None:
                if file_exists:
                    fieldnames = list(pd.read_csv(checkpoint_path, nrows=0).columns)
                else:
                    fieldnames = list(combined_row.keys())
            with open(checkpoint_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                if not file_exists:
                    writer.writeheader()
                writer.writerow(combined_row)

    results_by_idx = {}
    completed = 0
    iterator_kwargs = dict(total=len(pending)) if HAS_TQDM else {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_process_one_row, idx, row, client,
                             macro_model, financial_model, id_model, med_model): idx
            for idx, row in pending
        }

        progress = tqdm(as_completed(futures), **iterator_kwargs) if HAS_TQDM else as_completed(futures)

        for future in progress:
            idx, combined_row = future.result()
            write_row(combined_row)
            results_by_idx[idx] = combined_row
            completed += 1
            if not HAS_TQDM:
                print(f"[{completed}/{len(pending)}] {combined_row.get('filename')} "
                      f"Macro: {combined_row['macro_decision']} -> Leaf: {combined_row['final_subcategory']}")

    print(f"\nDone. {completed} document(s) processed this run. "
          f"Full results (including any resumed rows) are in '{checkpoint_path}'.")

    output_df = pd.read_csv(checkpoint_path)
    return output_df

# To run in your notebook cell:
# df_results = evaluate_experiment(
#     "test_data.csv",
#     "https://your-endpoint.openai.azure.com/",
#     "your-api-key",
#     macro_model="gpt-5-mini",
#     financial_model="gpt-5",
#     id_model="gpt-5",
#     med_model="gpt-5",
#     max_workers=5,
# )

In [ ]:
def evaluate_experiment(csv_path: str, endpoint: str, api_key: str,
                         triage_model: str = "gpt-5-mini",
                         nonmed_model: str = "gpt-5",
                         med_model: str = "gpt-5",
                         max_workers: int = 5,
                         checkpoint_path: str = "experiment_full_cascade_results.csv",
                         resume: bool = True):
    """Reads the evaluation data, executes the cascade pipeline concurrently, and
    returns results.
 
    triage_model / nonmed_model / med_model: deployment names for each stage. Defaults
        reflect a cost/performance split -- cheaper model where accuracy is already good
        (triage), stronger model where accuracy is the actual bottleneck (specialists).
        These must match deployment names that exist in your Azure resource, not the
        raw OpenAI model family names -- rename to whatever you deployed them as.
    max_workers: number of documents processed in parallel. Start conservative (3-5) and
        raise it while watching for [RATE LIMIT] log lines -- if you see frequent 429s
        even after backoff, max_workers is higher than your Azure deployment's TPM/RPM
        quota supports. Check your quota under Azure OpenAI Studio > Quotas.
    checkpoint_path: results are appended here as each document finishes, so a crash
        partway through doesn't lose completed work.
    resume: if True and checkpoint_path already exists, filenames already present there
        are skipped instead of being re-processed.
    """
 
    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version="2024-08-01-preview"
    )
 
    df = pd.read_csv(csv_path)
 
    already_done = set()
    if resume and os.path.isfile(checkpoint_path):
        prior = pd.read_csv(checkpoint_path)
        if 'filename' in prior.columns:
            already_done = set(prior['filename'].dropna().astype(str))
        print(f"Resuming: {len(already_done)} document(s) already in checkpoint, will be skipped.")
 
    pending = [
        (idx, row) for idx, row in df.iterrows()
        if str(row.get('filename')) not in already_done
    ]
 
    print(f"Starting cascade experiment on {len(pending)} document(s) "
          f"(of {len(df)} total, {len(already_done)} already done) with max_workers={max_workers}...")
 
    fieldnames = None  # inferred from the first completed row
 
    def write_row(combined_row):
        nonlocal fieldnames
        with _checkpoint_lock:
            file_exists = os.path.isfile(checkpoint_path)
            if fieldnames is None:
                if file_exists:
                    fieldnames = list(pd.read_csv(checkpoint_path, nrows=0).columns)
                else:
                    fieldnames = list(combined_row.keys())
            with open(checkpoint_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                if not file_exists:
                    writer.writeheader()
                writer.writerow(combined_row)
 
    results_by_idx = {}
    completed = 0
    iterator_kwargs = dict(total=len(pending)) if HAS_TQDM else {}
 
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_process_one_row, idx, row, client, triage_model, nonmed_model, med_model): idx
            for idx, row in pending
        }
 
        progress = tqdm(as_completed(futures), **iterator_kwargs) if HAS_TQDM else as_completed(futures)
 
        for future in progress:
            idx, combined_row = future.result()
            write_row(combined_row)
            results_by_idx[idx] = combined_row
            completed += 1
            if not HAS_TQDM:
                print(f"[{completed}/{len(pending)}] {combined_row.get('filename')} "
                      f"Triage: {combined_row['triage_decision']} -> Leaf: {combined_row['final_subcategory']}")
 
    print(f"\nDone. {completed} document(s) processed this run. "
          f"Full results (including any resumed rows) are in '{checkpoint_path}'.")
 
    output_df = pd.read_csv(checkpoint_path)
    return output_df
 
# To run in your notebook cell:
# df_results = evaluate_experiment(
#     "test_data.csv",
#     "https://your-endpoint.openai.azure.com/",
#     "your-api-key",
#     triage_model="gpt-5-mini",   # deployment name for the cheap/already-accurate stage
#     nonmed_model="gpt-5",        # deployment name for the accuracy bottleneck stage
#     med_model="gpt-5",           # swap to whatever your senior wants for medical
#     max_workers=5,                # tune to your Azure quota
# )